# 固定年度重训：原配置头与小配置头配对

只在每年首次决策前更新，冻结300轮与三seed，扩展历史；不能使用跨年未成熟标签。完整账户连续结算，不按年重置。

In [1]:
from pathlib import Path
import os,json
import qlib
from src.workflow.annual_allocator import run
root=Path.cwd()
while not (root/'SPEC.md').exists():root=root.parent
os.chdir(root);os.environ['MLFLOW_ALLOW_FILE_STORE']='true';card=root/'research/annual_allocator_refit_20261014';plan=json.loads((card/'plan.json').read_text());qlib.init(provider_uri=str(root/'data/qlib/full_path_tiered_allocation_20261013'),region='us',kernels=1)
summary=run(root=root,card=card,plan=plan)
print({k:summary[k] for k in ['decision','new_fits','new_native_cases','gates']})


In [1]:
from pathlib import Path
import os,json,hashlib
import numpy as np,pandas as pd,qlib
from qlib.workflow import R
from src.data.minute_intake import atomic_json
from src.data.nas_market_data import archive_nas_recorder
root=Path.cwd();card=root/'research/annual_allocator_refit_20261014';s=json.loads((card/'summary.json').read_text());qlib.init(provider_uri=s['provider_uri'],region='us',kernels=1);checks=[]
with R.uri_context(s['recorder_uri']):
 rec=R.get_recorder(recorder_id=s['recorder_id'],experiment_name=card.name)
 for family,reference in [('annual_original','frozen_original'),('annual_tangent','frozen_tangent')]:
  for cost in [0,2,5]:
   a=rec.load_object(f'analysis/native/{family}_{cost}')['authority'];b=rec.load_object(f'analysis/native/{reference}_{cost}')['authority'];mask=pd.to_datetime(a.trade_date).lt('2024-01-01');left=a.loc[mask,['equity','strategy_return']];right=b.loc[mask,['equity','strategy_return']];assert a.loc[mask,'trade_date'].equals(b.loc[mask,'trade_date']);np.testing.assert_allclose(left,right,rtol=1e-10,atol=1e-10);checks.append({'family':family,'slippage_bps':cost,'prefix_sessions':int(mask.sum()),'max_equity_difference':float((left.equity-right.equity).abs().max())})
 with R.start(experiment_name=card.name,recorder_id=s['recorder_id'],resume=True):
  rec=R.get_recorder();rec.save_objects(artifact_path='analysis',pre_refit_prefix_validation=pd.DataFrame(checks));rec.save_objects(artifact_path='inputs',operational_failure=json.loads((card/'operational_failure.json').read_text()));artifacts=Path(s['recorder_local_dir'])/'artifacts';rec.log_metrics(registered_artifact_count=sum(p.is_file() for p in artifacts.rglob('*')))
atomic_json({'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted(artifacts.rglob('*')) if p.is_file()]},card/'verification_manifest.json');s['qlib_docs']=['https://qlib.readthedocs.io/en/latest/component/data.html','https://qlib.readthedocs.io/en/latest/component/model.html'];s['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=s['experiment_id']);atomic_json(s,card/'summary.json');pd.DataFrame(checks).to_csv(card/'pre_refit_prefix_validation.csv',index=False)


# 固定年度扩展重训：两种结构均未改善完整净费账户

实际完成18次新拟合：2024/2025/2026各两种结构×3个固定seed，均300轮、固定最终轮与等权平均。2023按实际决策年复用6个已有模型。新跑6套完整连续账户，保留12套已验证对照。原结构与973→298参数的小配置头同时年度重训，不将更新数据的效果混作结构收益。

|首次使用年份|成熟主训练样本|辅助训练样本|原始观察截止|
|---|---|---|---|
|2024|3264|27991|2023-12-29 00:00:00|
|2025|3516|35318|2024-12-31 00:00:00|
|2026|3766|42818|2025-12-31 00:00:00|

每年纽约1月1日为信息截止，主标签必须已成熟；辅助原始观察先截断，再构造合成2x标签。使用年度由实际decision_at决定，不能按更早的signal_date分组。例如2023-12-29信号在2024-01-02决策，应使用2024模型，但该行未来标签不能进入其训练。训练/预测均通过Qlib DatasetH，辅助数据由D.features按截止日读取，源文件指纹前后相同。

预测按实际决策年拼接后连续平滑，整段账户只初始化一次，不跨年重置现金、仓位、风控高水位或月度份额计费。原动作/恢复控制器冻结不变。6个账户在2024重训启用前与各自冻结对照逐值一致；见pre_refit_prefix_validation.csv。年度绩效表沿用信号年归因，因此跨年末的一行可能使用次年模型，不能混淆模型切换与报表归因。

统一10万美元、冻结IBKR Pro Tiered费用投影、0/2/5bp独立状态。944个区间，2023-01-03信号起至2026-10-09结算。原模拟盘模型为统一口径重建，不是实际成交账户收益。

|模型|滑点|累计净收益|收盘最大回撤|盘中监控最大回撤|相对SOXX财富变化|
|---|---|---|---|---|---|
|原结构年度重训|0bp|+122.48%|-29.30%|-29.65%|-54.39%|
|小配置头年度重训|0bp|+271.10%|-24.93%|-26.33%|-23.92%|
|原结构冻结|0bp|+280.24%|-25.93%|-26.20%|-22.04%|
|小配置头冻结|0bp|+342.52%|-24.69%|-27.59%|-9.27%|
|原模拟盘模型重建|0bp|+228.99%|-28.11%|-29.50%|-32.55%|
|SOXX|0bp|+387.75%|-42.87%|-43.38%|+0.00%|
|原结构年度重训|2bp|+115.82%|-26.71%|-27.02%|-55.75%|
|小配置头年度重训|2bp|+260.80%|-24.94%|-26.35%|-26.02%|
|原结构冻结|2bp|+269.16%|-24.40%|-25.77%|-24.31%|
|小配置头冻结|2bp|+335.36%|-24.74%|-27.64%|-10.74%|
|原模拟盘模型重建|2bp|+279.67%|-24.17%|-27.07%|-22.16%|
|SOXX|2bp|+387.73%|-42.87%|-43.39%|+0.00%|
|原结构年度重训|5bp|+159.43%|-27.00%|-27.31%|-46.81%|
|小配置头年度重训|5bp|+255.23%|-24.99%|-26.41%|-27.16%|
|原结构冻结|5bp|+266.08%|-24.46%|-25.83%|-24.94%|
|小配置头冻结|5bp|+348.06%|-24.83%|-27.73%|-8.13%|
|原模拟盘模型重建|5bp|+41.12%|-31.38%|-31.42%|-71.06%|
|SOXX|5bp|+387.70%|-42.88%|-43.39%|+0.00%|

两种结构年度重训均在三个费用情景落后各自冻结对照。小模型2/5bp由冻结+335.36%/+348.06%降至+260.80%/+255.23%；原结构由+269.16%/+266.08%降至+115.82%/+159.43%。小结构仍优于同协议年度重训原结构，但这不能抵消更新协议本身的退步。

|信号年度|年度小模型2bp净收益|相对SOXX财富变化|年度小模型5bp净收益|相对SOXX财富变化|
|---|---|---|---|---|
|2023|+37.83%|-11.99%|+37.29%|-12.32%|
|2024|+30.23%|+7.65%|+29.64%|+7.16%|
|2025|+45.15%|-1.12%|+44.58%|-1.52%|
|2026|+38.49%|-21.03%|+38.05%|-21.29%|

相对财富为增长倍数之比减1，不是收益百分点。小模型年度重训满足30%回撤和相对SOXX的CVaR约束，但净超额、冻结小模型、原冻结模型、年度/滚动稳定性及区块置信门失败；独立开发和前向也未取得。仍为反复查看过的历史诊断，源版本是冻结回顾数据，不声称历史首次发布全部认证。

决定：not_promoted_fixed_annual_refit_failed。固定年度扩展协议关闭，不改为月度/季度、缩短滚动窗口或追加epoch挽救。结果只否定此网络/目标/冻结控制器下的固定更新协议，不证明所有重训方式无效。更近期数据与更多样本未自动转化为净收益；现役不变，稳定净超额目标仍未实现。

此前月度Ridge路线与本轮网络更新目标/结构不同；本轮没有重开那条失败映射。后续保留冻结小模型与原模型作强历史对照，新实验必须另有机制依据，不通过更新频率搜索选赢家。继续历史训练回测，不恢复实时等待。

执行中首次在活动Recorder内切换数据provider触发Qlib限制，0次拟合时即停止；修复为先准备辅助数据、后启动Recorder，原生回测在活动Recorder作用域外执行。科学计划未改变，失败记录见operational_failure.json。跨年时钟合同1 passed，实际模型的标签成熟/样本一致性、6组账户前缀一致性和NAS冷读回均验证。

[完整净费比较](backtest_benchmark_comparison.csv) · [训练审计](training_audit.csv) · [年度切换时钟](annual_clock.csv) · [区块区间](uncertainty.csv)

Recorder `326ee243e31b402bbd6540feb80765b6`。
